<a href="https://colab.research.google.com/github/sahanabalajee/MetaMuseum/blob/sofia/RS_Art_Gallery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


ValueError: mount failed

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

# Load MobileNetV2 for feature extraction
model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')

# Function to extract features
def extract_features(image_path, model):
    image = Image.open(image_path).convert('RGB').resize((224, 224))
    image_array = np.expand_dims(np.array(image), axis=0)
    image_array = preprocess_input(image_array)
    features = model.predict(image_array)
    return features.flatten()

# List images in folder
folder_path = '/content/drive/My Drive/art_gallery_dataset/original'  # Adjust the folder path
images = os.listdir(folder_path)

# Extract features for all images in the folder
image_features = {}
for img in images:
    img_path = os.path.join(folder_path, img)
    image_features[img] = extract_features(img_path, model)

# Calculate similarity matrix
feature_matrix = np.array(list(image_features.values()))
similarity_matrix = cosine_similarity(feature_matrix)

# Recommend similar images
def recommend_similar_images(input_image_path, model, image_features, top_n=5):
    input_features = extract_features(input_image_path, model)
    similarities = cosine_similarity([input_features], list(image_features.values()))[0]
    sorted_indices = np.argsort(similarities)[::-1][:top_n]
    similar_images = [list(image_features.keys())[i] for i in sorted_indices]
    return similar_images


In [ ]:
# Example input image
input_image_path = '/content/vict.jpeg'  # Adjust with an actual image path


In [ ]:

# Get recommended similar images
recommendations = recommend_similar_images(input_image_path, model, image_features)

# Display the recommended images
def display_images(image_paths, folder_path):
    plt.figure(figsize=(15, 15))
    for i, image_path in enumerate(image_paths):
        img = mpimg.imread(os.path.join(folder_path, image_path))
        plt.subplot(1, len(image_paths), i+1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(image_path)
    plt.show()

# Display the recommended images
display_images(recommendations, folder_path)